# 02 · IBD Intake & QC (Beginner‑Friendly)

**Goal:** Load GSE235236 counts + metadata via manifest. Check shapes, missing values,
group labels (Control/UC/CD), and save a clean version for downstream.


In [ ]:

BASE = "/content/drive/MyDrive/Colorectal_Hippo_Dysbiosis"
MANIFEST = f"{BASE}/config/manifest_ibd.csv"
print("Using manifest:", MANIFEST)


In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, re
from pathlib import Path

def read_manifest(path):
    df = pd.read_csv(path)
    assert set(["key","path"]).issubset(df.columns), "Manifest must have columns: key,path"
    return df.set_index("key")["path"].to_dict()

def guess_counts_orientation(df):
    tmp = df.copy()
    if tmp.columns[0].lower() in ["gene","gene_id","symbol"]:
        tmp = tmp.set_index(tmp.columns[0])
    if tmp.shape[0] > tmp.shape[1] and np.issubdtype(tmp.dtypes.min(), np.number):
        return 'genes_by_cols', tmp.T
    return 'genes_by_rows', tmp

def clean_ids(idx):
    return pd.Index(idx.astype(str).str.strip().str.replace(r'[^A-Za-z0-9_\-\.]+','_', regex=True))


In [ ]:

man = read_manifest(MANIFEST); man


In [ ]:

print("\n=== Loading counts ===")
counts_raw = pd.read_csv(man["counts"], sep=None, engine="python")
orient, counts = guess_counts_orientation(counts_raw)
print("Orientation:", orient)
print("Counts shape (genes x samples):", counts.shape)
display(counts.head())

print("Min value:", counts.min().min())
print("Total NA:", counts.isna().sum().sum())


In [ ]:

print("\n=== Loading metadata ===")
meta = pd.read_csv(man["metadata"])
print("Metadata shape:", meta.shape)
display(meta.head())

candidates = [c for c in meta.columns if c.lower() in ["group","status","condition","phenotype","disease","diagnosis"]]
print("Candidate group columns:", candidates)
group_col = candidates[0] if candidates else meta.columns[-1]

# Clean IDs & infer sample id column
counts.columns = clean_ids(counts.columns)
id_cols = [c for c in meta.columns if re.search("sample|run|gsm|id", c, flags=re.I)]
sid = id_cols[0] if id_cols else meta.columns[0]
meta[sid] = clean_ids(meta[sid])

meta_keep = meta[meta[sid].isin(counts.columns)].copy()
print(f"Overlap samples: {meta_keep.shape[0]} / {counts.shape[1]}")
if group_col in meta_keep.columns:
    print("Group sizes:"); display(meta_keep[group_col].value_counts())


In [ ]:

# Library sizes
libsize = counts.sum(axis=0)
plt.figure(figsize=(6,4))
plt.hist(libsize, bins=30)
plt.title("Library size per sample (IBD)")
plt.xlabel("Total counts"); plt.ylabel("Samples")
plt.tight_layout(); plt.show()


In [ ]:

counts.to_csv(f"{BASE}/data_processed/ibd_counts_clean.csv")
meta_keep.to_csv(f"{BASE}/data_processed/ibd_metadata_clean.csv", index=False)
print("Saved -> data_processed/ibd_counts_clean.csv")
print("Saved -> data_processed/ibd_metadata_clean.csv")
print("\nNEXT → Run 03_norm_batch_merge.ipynb")
